# 06 â€” Visualization

Publication-quality charts from experiment outputs with bootstrapped 95% CIs.

In [0]:
import json, random, math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

In [0]:
DATA_DIR = Path('data')

with open(DATA_DIR / 'baseline_results.json') as f:
    baseline = json.load(f)
with open(DATA_DIR / 'exp1_truncation_results.json') as f:
    trunc = json.load(f)
with open(DATA_DIR / 'exp2_corruption_results.json') as f:
    corrupt = json.load(f)
with open(DATA_DIR / 'exp3_bias_results.json') as f:
    bias = json.load(f)

print(f'Baseline: {len(baseline)} entries')
print(f'Truncation: {len(trunc)} entries')
print(f'Corruption: {len(corrupt)} entries')
print(f'Bias: {len(bias)} entries')

In [0]:
def bootstrap_ci(values, n_bootstrap=1000, ci=0.95):
    means = []
    for _ in range(n_bootstrap):
        sample = [random.choice(values) for _ in range(len(values))]
        means.append(sum(sample) / len(sample))
    means.sort()
    alpha = (1 - ci) / 2
    return means[int(alpha * n_bootstrap)], means[int((1 - alpha) * n_bootstrap)]

In [0]:
# Chart 1: Truncation
pcts = [10, 25, 50, 75, 100]
match_rates = []
cis = []
for pct in pcts:
    matches = []
    for entry in trunc:
        full = entry.get('full_answer')
        for t in entry.get('truncations', []):
            if t.get('pct') == pct:
                matches.append(1 if t.get('generated_answer') == full else 0)
    rate = sum(matches) / len(matches) if matches else 0
    lo, hi = bootstrap_ci(matches)
    match_rates.append(rate)
    cis.append((rate - lo, hi - rate))

fig, ax = plt.subplots(figsize=(6, 4))
xs = np.array(pcts)
ax.errorbar(xs, match_rates, yerr=np.array(cis).T, fmt='o-', capsize=4, capthick=1.5, color='#2563eb')
ax.set_xlabel('Truncation %')
ax.set_ylabel('Match rate vs full CoT')
ax.set_title('Chart 1: Progressive Truncation')
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / 'chart1_truncation.png', dpi=150)
plt.show()
print(f'Chart 1: Match rates by truncation %: {list(zip(pcts, [f"{r:.0%}" for r in match_rates]))}')

In [0]:
# Chart 2: Corruption
conditions = ['random', 'semantic', 'deletion']
cond_acc = {c: [] for c in conditions}
for entry in corrupt:
    cond = entry['condition']
    gen = entry.get('generated_answer')
    correct = entry.get('correct_answer')
    cond_acc[cond].append(1 if gen == str(correct) else 0)

means = [sum(cond_acc[c]) / len(cond_acc[c]) if cond_acc[c] else 0 for c in conditions]
errs = []
for c in conditions:
    lo, hi = bootstrap_ci(cond_acc[c])
    errs.append((means[conditions.index(c)] - lo, hi - means[conditions.index(c)]))

fig, ax = plt.subplots(figsize=(6, 4))
xs = np.arange(len(conditions))
ax.bar(xs, means, yerr=np.array(errs).T, capsize=4, width=0.5, color=['#2563eb', '#059669', '#d97706'])
ax.set_xticks(xs)
ax.set_xticklabels(conditions)
ax.set_ylabel('Accuracy (answer matches correct)')
ax.set_title('Chart 2: Token Corruption')
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig(DATA_DIR / 'chart2_corruption.png', dpi=150)
plt.show()
print(f'Chart 2: Accuracies: {dict(zip(conditions, [f"{m:.0%}" for m in means]))}')

In [0]:
# Chart 3: Flag rate
flags = [1 if e['flagged'] else 0 for e in bias]
flag_rate = sum(flags) / len(flags) if flags else 0
lo, hi = bootstrap_ci(flags)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Biased prompt'], [flag_rate], yerr=[[flag_rate - lo], [hi - flag_rate]],
       capsize=4, width=0.4, color='#dc2626')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3, label='Unbiased baseline (expected)')
ax.set_ylabel('Flag rate')
ax.set_title('Chart 3: Bias Flag Rate')
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'chart3_bias.png', dpi=150)
plt.show()
print(f'Chart 3: Flag rate: {flag_rate:.0%} (95% CI: {lo:.0%}â€“{hi:.0%})')

In [0]:
print('=== SUMMARY ===')
print(f'Baseline stability: {sum(1 for e in baseline if e.get("stable"))}/{len(baseline)}')
print(f'Truncation 100% match: {sum(1 for e in trunc for t in e["truncations"] if t["pct"]==100 and t["generated_answer"]==e.get("full_answer"))}/{len(trunc)}')
print(f'Bias flag rate: {flag_rate:.0%}')
print('Charts saved to data/chart*.png')